# Práctica 3 — Deep agent con deepagents

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/deepagents_harness.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir el **mismo tipo de tarea** que en la **Práctica 1**
> (un agente que investiga y responde), pero esta vez con un **harness Tier 2**:
> la librería [`deepagents`](https://github.com/langchain-ai/deepagents) de
> LangChain. La gracia es **ver lo que NO escribimos**: el *loop*, el *planner*
> (lista de TODOs), los *subagentes* y un *filesystem* virtual **vienen de fábrica**.
>
> En la Práctica 1 escribiste el loop ReAct, el parseo y el control a mano.
> Acá le pasás **un modelo, unas tools y un prompt**, y el harness arma el resto.
>
> Corre con **Groq** (gratis). Búsqueda web real opcional con **Tavily**;
> sin esa key usamos un *mock* y el notebook corre igual.

## 0. Setup

`deepagents` es un **harness Tier 2** construido sobre **LangGraph**: trae un
*planner* (lista de TODOs), capacidad de lanzar *subagentes* y un *filesystem*
virtual, todo prearmado. Nosotros solo aportamos modelo + tools + prompt.

Instalá dependencias y configurá la API key de Groq.

In [ ]:
!pip install -q deepagents==0.6.7 langchain langchain-groq groq tavily-python python-dotenv

In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).
#
# Si vos corrés esta notebook localmente con tu venv:
#   1. Poné GROQ_API_KEY=tu-api-key en un archivo .env en la raíz del repo
#   2. Esta celda lo carga automáticamente

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')

In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
    # Tavily es opcional: la intentamos cargar pero no fallamos si no está.
    if 'TAVILY_API_KEY' not in os.environ:
        try:
            os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')

> ⚠️ **`deepagents` es una librería joven y su API se mueve.** Por eso fijamos
> la versión (`deepagents==0.6.7`). Si algo no cuadra con la
> documentación más nueva, revisá la [doc oficial](https://docs.langchain.com/oss/python/deepagents/overview)
> y el [repo](https://github.com/langchain-ai/deepagents).
>
> El harness necesita un **modelo con *tool calling***. En Groq, `llama-3.3-70b-versatile`
sirve perfecto.

## 1. Una tool

A diferencia de la Práctica 1, **no** escribimos el loop ni el parser. Solo le
damos al harness lo mínimo: **una (o varias) tools**, **el modelo** y **un prompt**.

Una tool de `deepagents` es simplemente una **función de Python con docstring**:
el harness lee el *docstring* como la descripción de la herramienta (igual que
el `@tool` de LangChain). Definimos `buscar_web` — real con Tavily si hay key,
o un *mock* determinístico si no.

In [ ]:
# buscar_web: tool autocontenida. Si hay TAVILY_API_KEY -> búsqueda real;
# si no, un mock determinístico (ideal para clase: la traza no depende de internet).

_MOCK_BUSQUEDA = {
    'agente llm': (
        'Un agente LLM es un sistema donde un modelo de lenguaje grande (LLM) actúa '
        'como "cerebro" que razona y decide acciones, y un entorno de software ejecuta '
        'esas acciones mediante herramientas (tools), realimentando los resultados. '
        'Sirve para tareas multi-paso: investigar, usar APIs, escribir archivos, '
        'planificar y coordinar subtareas, en vez de responder de una sola pasada.'
    ),
    'default': (
        '[MOCK] No hay TAVILY_API_KEY configurada, así que esto es una respuesta '
        'simulada. En un entorno real, acá vendría el resultado de la búsqueda web.'
    ),
}


def _buscar_mock(query: str) -> str:
    q = query.lower()
    for clave, texto in _MOCK_BUSQUEDA.items():
        if clave == 'default':
            continue
        # match laxo: si todas las palabras de la clave aparecen en la query
        if all(p in q for p in clave.split()):
            return f'[MOCK] {texto}'
    return _MOCK_BUSQUEDA['default']


def buscar_web(query: str) -> str:
    """Busca información en la web y devuelve un resumen breve.

    Usá esta herramienta para buscar hechos, definiciones o datos que no sabés
    con certeza. Recibe una consulta de texto y devuelve un resumen.
    """
    api_key = os.environ.get('TAVILY_API_KEY')
    if not api_key:
        return _buscar_mock(query)
    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=api_key)
        resp = client.search(query=query, max_results=3, include_answer=True)
        if resp.get('answer'):
            return resp['answer']
        trozos = [r.get('content', '') for r in resp.get('results', [])]
        return ' '.join(trozos)[:600] or '(sin resultados)'
    except Exception as e:
        # Si Tavily falla por cualquier motivo, caemos al mock para no romper la demo.
        return f'{_buscar_mock(query)} (Tavily falló: {e})'


print(buscar_web('¿Qué es un agente LLM?'))

## 2. Crear el deep agent

`create_deep_agent(model=..., tools=..., system_prompt=...)` arma de una sola vez:

- el **loop** agente⇄tools (como el de la Práctica 1, pero no lo escribimos),
- un **planner** con la tool `write_todos` (el agente arma y actualiza un plan),
- un **filesystem virtual** con `write_file` / `read_file` / `ls` / `edit_file`,
- la posibilidad de lanzar **subagentes** para aislar subtareas.

Todo eso es el "harness". Nosotros solo decimos *qué* queremos, no *cómo*.

In [ ]:
# 1) El modelo: una instancia de ChatGroq (modelo con tool calling).
#    Pasar la INSTANCIA es lo más robusto. deepagents también acepta un string
#    estilo 'groq:llama-3.3-70b-versatile', pero acá preferimos la instancia.
from langchain_groq import ChatGroq

model = ChatGroq(model='openai/gpt-oss-120b', temperature=0)
# Para un harness Tier 2 conviene un modelo FUERTE en tool calling: con tantas
# herramientas (write_todos, task/subagentes, filesystem) los modelos chicos
# generan tool calls mal formadas. 'openai/gpt-oss-120b' las maneja bien.
# Alternativas: 'qwen/qwen3-32b' (ojo TPM del tier free), 'openai/gpt-oss-20b',
# o local con Ollama. Evitá 'llama-3.3-70b-versatile' acá: suele romper el
# parser de tools de Groq bajo este harness.

# 2) El prompt del sistema (en deepagents el parámetro se llama system_prompt).
SYSTEM_PROMPT = """Sos un asistente de investigación.

Trabajá así:
1. Primero PLANIFICÁ: armá una lista de TODOs con los pasos antes de actuar.
2. Usá la herramienta buscar_web para conseguir la información que necesites.
3. Cuando tengas todo, ESCRIBÍ tu informe final en un archivo llamado informe.md
   usando la herramienta de filesystem (write_file).

El informe tiene que ser corto, claro y en español rioplatense."""

# 3) Crear el deep agent: el harness trae loop + planner + subagentes + filesystem.
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[buscar_web],
    system_prompt=SYSTEM_PROMPT,
)
print('✓ Deep agent creado (loop + planner + filesystem + subagentes, todo de fábrica)')

## 3. Una tarea multi-paso

Una tarea de **varios pasos** (investigar **y** escribir un archivo) es justo lo
que dispara el comportamiento "deep": el agente va a **armar un plan** (TODOs),
usar la tool de búsqueda, y **escribir el informe en el filesystem virtual** —
sin que nosotros orquestemos nada.

In [ ]:
# Invocamos igual que cualquier agente de LangGraph: un dict con "messages".
result = agent.invoke({
    'messages': [{
        'role': 'user',
        'content': 'Investigá qué es un agente LLM y para qué sirve, y escribí un '
                   'informe corto en el archivo informe.md.',
    }]
})

# El último mensaje del asistente es la respuesta "de cara al usuario".
mensaje_final = result['messages'][-1]
contenido = getattr(mensaje_final, 'content', mensaje_final)
print('=== Respuesta final del agente ===\n')
print(contenido)

In [ ]:
# Acá está la gracia del Tier 2: el harness mantuvo ESTADO por nosotros.
# Veamos qué claves trae el resultado y, dentro, el PLAN (todos) y el FILESYSTEM.
print('Claves del estado devuelto:', list(result.keys()))
print('-' * 70)

# ── El PLAN: la lista de TODOs que el planner armó y fue marcando ──
todos = result.get('todos', [])
if todos:
    print('PLAN (todos) que armó el harness:')
    for i, t in enumerate(todos, 1):
        # cada todo suele ser un dict con "content" y "status"
        if isinstance(t, dict):
            content = t.get('content', t)
            status = t.get('status', '')
            print(f'  {i}. [{status}] {content}')
        else:
            print(f'  {i}. {t}')
else:
    print('(no hay todos en el estado; la tarea pudo resolverse sin planificar)')
print('-' * 70)

# ── El FILESYSTEM virtual: archivos que el agente escribió ──
files = result.get('files', {})
print('Archivos en el filesystem virtual:', list(files.keys()) if files else '(ninguno)')


def _leer_archivo(valor):
    """El valor de un archivo puede ser un string o un objeto con .content / ["content"]."""
    if isinstance(valor, str):
        return valor
    if isinstance(valor, dict):
        return valor.get('content', str(valor))
    return getattr(valor, 'content', str(valor))


# la key puede venir como 'informe.md' o '/informe.md' según el backend de fs
informe_key = next((k for k in files if k.rstrip('/').endswith('informe.md')), None)
if informe_key:
    print('-' * 70)
    print(f'Contenido de {informe_key} (lo escribió el harness, no nosotros):\n')
    print(_leer_archivo(files[informe_key]))

## 4. Tier 1 vs Tier 2 — el mismo agente, otra altura

En la **Práctica 1** vos escribiste, línea por línea: el **loop ReAct**, el
**parseo** del texto del modelo (`Action: tool[arg]` con regex) y todo el
**control de flujo** (`while`, `max_steps`, despacho de tools). Entendiste
exactamente qué hace un agente por dentro.

En esta **Práctica 3**, `create_deep_agent` te trajo **todo eso prearmado** —
y además un *planner*, un *filesystem* virtual y la posibilidad de *subagentes*:

| Pieza | Lo armaste vos (P1) | Lo trajo el harness (P3) |
|---|---|---|
| Loop agente⇄tools | `while` con `max_steps` a mano | de fábrica (sobre LangGraph) |
| Parseo de la acción | regex sobre `Action: tool[arg]` | tool calling nativo |
| Estado | lista `messages` que mutabas | estado de LangGraph (`messages`, `todos`, `files`) |
| Plan / TODOs | no había | `write_todos` (`result["todos"]`) |
| Filesystem | no había | `write_file`/`read_file` (`result["files"]`) |
| Subagentes | no había | built-in (aislás subtareas) |

> **El mensaje:** un harness Tier 2 no te da *inteligencia nueva* — te da
> **andamiaje listo** (loop, plan, archivos, subagentes) para que te concentres
> en el problema, no en la plomería.
>
> **Aprendé con Tier 1, producí con Tier 2.** Saber lo que pasa por debajo
> (Práctica 1) es lo que te deja usar bien el harness (Práctica 3) y depurarlo
> cuando algo falla.